# 1. Working with Sequences and JSON
## Create a program that:
• Reads a JSON file containing a list of students and their grades.
• Sorts the students by their average grade. 
• Compares their scores using Python sequence methods.
• Writes the sorted list to a new JSON file.
• Include exception handling for malformed files and missing data.

In [2]:
import json

def average_grade(grades):
    """Calculates average grade with error handling"""
    try:
        return sum(grades) / len(grades)
    except KeyError as e:
        raise ValueError(f"Invalid grade found: {e}")

try:
    """Read students from JSON file"""
    with open('students.json', 'r') as file:
        students = json.load(file)
                            
    """Calculate averages of each student and add to each student respectively"""
    for student in students:
        if "grades" not in student or not student["grades"]:
            raise ValueError(f"Missing or empty grades for student: {student.get('name')}")
        student["average"] = average_grade(student["grades"])

    """Sort the students by their average grade in descending order"""
    sorted_students = sorted(students, key=lambda s: s["average"], reverse=True)
    
    """Compare grades using sequence methods"""
    top_student = sorted_students[0]
    print(f"Top student: {top_student['name']} with grades {top_student['grades']}")
    print("Grades sorted of the top student in descending order:", sorted(top_student['grades'], reverse=True))
    print("Highest grade of a student:", max(student['average'] for student in students))
    print("Lowest grade of a student:", min(student['average'] for student in students))
    
    """Write sorted students to new JSON file"""
    with open('sorted_students.json', 'w') as outfile:
        json.dump(sorted_students, outfile, indent=4)

except json.JSONDecodeError:
    print("Error: JSON file is malformed.")
except FileNotFoundError:
    print("Error: students.json file not found.")
except ValueError as ve:
    print("Data error:", ve)
except Exception as e:
    print("An unexpected error occurred:", e)

Top student: Charlie with grades [95, 100, 98, 97]
Grades sorted of the top student in descending order: [100, 98, 97, 95]
Highest grade of a student: 97.5
Lowest grade of a student: 68.75


# 2. Exception Handling and User-defined Exceptions
## Develop a banking application that:
• Allows users to deposit and withdraw money.
• Raises and handles exceptions for:
o Insufficient funds
o Negative input
o Incorrect account operations
• Uses a user-defined exception class for handling custom banking errors.

In [4]:
class BankingError(Exception):
    """Base class for all banking exceptions"""
    pass

class InsufficientFundsError(BankingError):
    pass

class NegativeInputError(BankingError):
    pass

class IncorrectOperationError(BankingError):
    pass

class BankAccount:
    def __init__(self, account_number, holder_name, balance=0):
        self.account_number = account_number
        self.holder_name = holder_name
        self.balance = balance

    def deposit(self, amount):
        """Handle deposits with validation"""
        if amount <= 0:
            raise NegativeInputError("Deposit amount must be positive")
        self.balance += amount
        return self.balance

    def withdraw(self, amount):
        """Handle withdrawals with validation"""
        if amount <= 0:
            raise NegativeInputError("Withdrawal amount must be positive")
        if amount > self.balance:
            raise InsufficientFundsError("Insufficient funds for withdrawal")
        self.balance -= amount
        return self.balance

    def transfer(self, amount, to_account):
        """Handle transfers between accounts"""
        if self.account_number == to_account.account_number:
            raise IncorrectOperationError("Cannot transfer to the same account")
        
        # Withdraw from this account first
        self.withdraw(amount)
        
        # If withdrawal succeeds, deposit to target account
        to_account.deposit(amount)
        return True

if __name__ == "__main__":
    try:
        # Create test accounts
        acc1 = BankAccount("123456", "Alice", 1000)
        acc2 = BankAccount("654321", "Bob", 500)

        # Valid operations
        acc1.deposit(500)
        acc1.withdraw(200)
        acc1.transfer(300, acc2)

        # Trigger exceptions (uncomment to test)
        # acc1.deposit(-100)          # NegativeInputError
        # acc1.withdraw(2000)         # InsufficientFundsError
        # acc1.transfer(500, acc1)    # IncorrectOperationError

    except NegativeInputError as e:
        print(f"Invalid amount: {e}")
    except InsufficientFundsError as e:
        print(f"Funds error: {e}")
    except IncorrectOperationError as e:
        print(f"Operation error: {e}")
    except BankingError as e:
        print(f"Banking error: {e}")

    # Print final balances
    print(f"\nAlice's balance: ₹{acc1.balance}")
    print(f"Bob's balance: ₹{acc2.balance}")



Alice's balance: ₹1000
Bob's balance: ₹800


# 3. File Operations and Object-Oriented Design
## Write a class-based program that:
• Reads from and writes to a CSV file.
• Encapsulates file operations in class methods.
• Defines a FileManager class with:
o Private variables
o Error handling
o Method documentation
• Demonstrates inheritance for handling different file types (text, CSV).

In [6]:
import csv

class FileManager:
    """Base class for file operations with error handling"""
    def __init__(self, filename):
        self.__filename = filename

    def read_file(self):
        """Reads content from the file"""
        try:
            with open(self.__filename, 'r') as file:
                return file.read()
        except FileNotFoundError:
            return f"Error: File '{self.__filename}' not found."
        except Exception as e:
            return f"Error reading file: {str(e)}"

    def write_file(self, data):
        """Writes data to the file"""
        try:
            with open(self.__filename, 'w') as file:
                file.write(data)
            return f"Data written to '{self.__filename}' successfully." 
        except Exception as e:
            return f"Error writing to file: {str(e)}"

class CSVFileManager(FileManager):
    """Handles CSV files with DictReader/DictWriter"""
    def read_file(self):
        try:
            with open(self._FileManager__filename, 'r') as file:
                return list(csv.DictReader(file))
        except FileNotFoundError:
            return super().read_file()
        except Exception as e:
            return f"CSV read error: {str(e)}"

    def write_file(self, data):
        if not data:
            return "No data provided to write."
        try:
            with open(self._FileManager__filename, 'w', newline='') as file:
                fieldnames = data[0].keys()  # Corrected from search results
                writer = csv.DictWriter(file, fieldnames=fieldnames)
                writer.writeheader()
                writer.writerows(data)
            return f"CSV data written to '{self._FileManager__filename}' successfully."
        except Exception as e:
            return f"CSV write error: {str(e)}"

class TextFileManager(FileManager):
    """Handles text files using base class methods"""
    pass  # Inherits all functionality from FileManager

# Demonstration
if __name__ == '__main__':
    # Text file operations
    text_manager = TextFileManager('diary.txt')
    print(text_manager.write_file("Today's log: System operational"))
    print(text_manager.read_file())

    # CSV file operations
    csv_manager = CSVFileManager('data.csv')
    print(csv_manager.write_file([
        {'Employee': 'Alice', 'Salary': '95000'},
        {'Employee': 'Bob', 'Salary': '87000'}
    ]))
    print(csv_manager.read_file())


Data written to 'diary.txt' successfully.
Today's log: System operational
CSV data written to 'data.csv' successfully.
[{'Employee': 'Alice', 'Salary': '95000'}, {'Employee': 'Bob', 'Salary': '87000'}]


# 4. Iterators and Generators
## Build a custom iterable class that:
• Implements __iter__() and __next__() to generate Fibonacci numbers up to a limit.
• Also includes:
o A generator function
o A generator expression
• Compares performance and code readability between the three approaches.

In [8]:
import timeit
from itertools import islice, takewhile

# 1. Custom Iterable Class
class FibonacciIterator:
    """Class-based iterator for Fibonacci sequence up to a limit"""
    def __init__(self, limit):
        self.limit = limit
        self.a = 0
        self.b = 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.a > self.limit:
            raise StopIteration
        current = self.a
        self.a, self.b = self.b, self.a + self.b
        return current

# 2. Generator Function
def fib_generator(limit):
    """Generator function for Fibonacci sequence up to limit"""
    a, b = 0, 1
    while a <= limit:
        yield a
        a, b = b, a + b

# 3. Generator Expression
def fib_sequence():
    """Generator Expression for Fibonacci sequence up to limit"""
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

def generate_fib_up_to(limit):
    return takewhile(lambda x: x <= limit, fib_sequence())

# Performance Comparison
iterations = 10000
limit = 1000

class_time = timeit.timeit(
    'list(FibonacciIterator(limit))',
    globals=globals(),
    number=iterations
)

func_time = timeit.timeit(
    'list(fib_generator(limit))',
    globals=globals(),
    number=iterations
)

expr_time = timeit.timeit(
    'list(generate_fib_up_to(21))',
    globals=globals(),
    number=iterations
)

# Readability Comparison
readability = """
| Approach             | Readability Score | Maintenance Score |
|----------------------|-------------------|-------------------|
| Class Iterator       | Moderate          | High              |
| Generator Function   | High              | High              |
| Generator Expression | Low               | Low               |
"""

# Demonstration
if __name__ == "__main__":
    print("Class Iterator:", list(FibonacciIterator(21)))
    print("Generator Func:", list(fib_generator(21)))
    print("Generator Expr:", list(generate_fib_up_to(21)))
    
    print("\nPerformance Results (10000 iterations):")
    print(f"Class: {class_time:.4f}s")
    print(f"Function: {func_time:.4f}s")
    print(f"Expression: {expr_time:.4f}s")
    
    print("\nReadability Comparison:")
    print(readability)
    print("- Iterator Class  ➜ Most flexible, reusable, slightly more verbose.")
    print("- Generator Func  ➜ Easiest to read & maintain. Great for most use cases.")
    print("- Gen Expression  ➜ Most concise, but less intuitive if logic gets complex.")

Class Iterator: [0, 1, 1, 2, 3, 5, 8, 13, 21]
Generator Func: [0, 1, 1, 2, 3, 5, 8, 13, 21]
Generator Expr: [0, 1, 1, 2, 3, 5, 8, 13, 21]

Performance Results (10000 iterations):
Class: 0.0160s
Function: 0.0074s
Expression: 0.0088s

Readability Comparison:

| Approach             | Readability Score | Maintenance Score |
|----------------------|-------------------|-------------------|
| Class Iterator       | Moderate          | High              |
| Generator Function   | High              | High              |
| Generator Expression | Low               | Low               |

- Iterator Class  ➜ Most flexible, reusable, slightly more verbose.
- Generator Func  ➜ Easiest to read & maintain. Great for most use cases.
- Gen Expression  ➜ Most concise, but less intuitive if logic gets complex.


# 5. Advanced Class Design and Multiple Inheritance
Design a role-based access system:
• Use classes like User, Admin, Guest, with shared and specific functionality.
• Implement multiple inheritance where appropriate.
• Use class and instance variables to track user states.
• Override methods and demonstrate polymorphism.
• Include documentation strings and error-handling features.

In [10]:
class User:
    """Base class for all users in the system."""
    user_count = 0  # class variable to track total users

    def __init__(self, username):
        self.username = username
        self.logged_in = False  # instance variable
        User.user_count += 1

    def login(self):
        self.logged_in = True
        print(f"{self.username} logged in.")

    def logout(self):
        self.logged_in = False
        print(f"{self.username} logged out.")

    def access(self):
        """Override this in child classes"""
        raise NotImplementedError("Access method must be implemented by subclasses.")

    def __str__(self):
        return f"User: {self.username} (Logged in: {self.logged_in})"


class Guest(User):
    """Guest users with read-only access."""

    def access(self):
        if not self.logged_in:
            raise PermissionError("Guest must be logged in to access resources.")
        return f"Guest {self.username} has read-only access."


class Admin(User):
    """Admin users with full access."""

    def access(self):
        if not self.logged_in:
            raise PermissionError("Admin must be logged in to access resources.")
        return f"Admin {self.username} has full access."

    def delete_user(self, user):
        if not isinstance(user, User):
            raise ValueError("Can only delete User objects.")
        print(f"Admin {self.username} deleted user {user.username}.")


class AuditLogger:
    """Mixin class to log actions for auditing."""

    def log_action(self, action):
        print(f"[AUDIT] {self.username}: {action}")


class SuperAdmin(Admin, AuditLogger):
    """SuperAdmin has admin rights and logs every action."""

    def access(self):
        self.log_action("Access attempt")
        return super().access()

    def delete_user(self, user):
        self.log_action(f"Attempted to delete user: {user.username}")
        super().delete_user(user)

    def shutdown_system(self):
        self.log_action("System shutdown initiated.")
        print(f"SuperAdmin {self.username} shut down the system.")


def main():
    print("Role-Based Access System Demo\n")

    guest = Guest("guest_user")
    admin = Admin("admin_user")
    super_admin = SuperAdmin("superadmin_user")

    try:
        guest.login()
        print(guest.access())
        guest.logout()

        admin.login()
        print(admin.access())
        admin.logout()

        super_admin.login()
        print(super_admin.access())
        super_admin.delete_user(guest)
        super_admin.shutdown_system()
        super_admin.logout()

    except Exception as e:
        print(f" Error: {e}")

    print(f"\nTotal users created: {User.user_count}")

if __name__ == "__main__":
    main()


Role-Based Access System Demo

guest_user logged in.
Guest guest_user has read-only access.
guest_user logged out.
admin_user logged in.
Admin admin_user has full access.
admin_user logged out.
superadmin_user logged in.
[AUDIT] superadmin_user: Access attempt
Admin superadmin_user has full access.
[AUDIT] superadmin_user: Attempted to delete user: guest_user
Admin superadmin_user deleted user guest_user.
[AUDIT] superadmin_user: System shutdown initiated.
SuperAdmin superadmin_user shut down the system.
superadmin_user logged out.

Total users created: 3
